## Single Agent 

In [50]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
import requests
from langchain.tools import tool

In [36]:
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_API_KEY = os.getenv("WEATHER_API_KEY")

In [37]:
#======================================
# Internet search tool
#======================================
search_tool = TavilySearchResults(max_results=3)


In [38]:
search_tool.invoke("what is the latest news in india")

[{'title': "India News, Latest India News, Today's Breaking News Headlines from India | The Indian Express",
  'url': 'https://indianexpress.com/section/india',
  'content': "September 9, 2026 03:06 IST\n\nForest officer raises smuggling concerns: 'Chances of their natural migration to the Balasore forest is less as there is no corridor link. It must be a human-induced act'\n\nhaldwani violence\n\n## Tear-gas squads, drones: Uttarakhand town preps for Supreme Court hearing\n\nSeptember 8, 2026 13:38 IST\n\nThe case in the top court concerns the land around Haldwani railway station, covering the localities of Gafoor Basti, Dholak Basti and Indira Nagar.\n\nIsro Earth Observation Satellite EOS-05\n\n## New ‘eye in the sky’: Isro’s EOS-05 satellite settles into geosynchronous orbit\n\nSeptember 8, 2026 12:28 IST\n\nOver the next few days, the EOS-05 satellite will undergo health and systems checks before its imaging systems are activated. [...] Kashmir: Intern doctors hold placards during

In [39]:
#=====================================
# LLM
#=====================================
llm = ChatGroq(api_key=GROQ_API_KEY,model='openai/gpt-oss-20b')

In [40]:
llm.invoke('what year is it')

AIMessage(content='It’s currently the year\u202f2026.', additional_kwargs={'reasoning_content': 'The user asks: "what year is it". They might want the current year. The system says current date is 2026-09-09. So answer: 2026. Probably mention date.'}, response_metadata={'token_usage': {'completion_tokens': 61, 'prompt_tokens': 75, 'total_tokens': 136, 'completion_time': 0.079761808, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.00468701, 'prompt_tokens_details': None, 'queue_time': 0.314790667, 'total_time': 0.084448818}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a085cd-9de0-7b41-8896-1946afc2edd6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 75, 'output_tokens': 61, 'total_tokens': 136, 'output_token_details': {'reasoning': 43}})

In [41]:
tools = [search_tool]

In [42]:
agent = create_agent(model=llm,tools=tools,system_prompt='you are helpfull asistant')

In [43]:
response = agent.invoke({"messages":[HumanMessage(content='find the captial of india then find its current weather')]})

In [44]:
print(response['messages'][-1].content)


**Capital of India**  
- **New Delhi**

**Current weather in New Delhi** (as of the latest live update on 9 Sep 2026)

| Parameter | Value | Source |
|-----------|-------|--------|
| Temperature | ~29 °C (≈84 °F) | Indian Express “New Delhi Weather Today” |
| Weather condition | Clear / Sunny | Same source |
| Humidity | ~48 % | Same source |
| Wind | ~7.9 km/h (≈5 mph) from the west | Same source |
| UV Index | 0 (low) | Same source |

> *Note:* The figures are pulled from a real‑time weather snapshot available on the Indian Express website. Weather can change quickly, so for the most up‑to‑date reading you may want to check a live weather app or the official meteorological service.

**Quick reference link**  
- [New Delhi Weather – Indian Express (live update)](https://indianexpress.com/section/weather/new-delhi-weather-forecast-today)


In [45]:
response


{'messages': [HumanMessage(content='find the captial of india then find its current weather', additional_kwargs={}, response_metadata={}, id='8de89e56-7336-4778-8556-f5a13d2c86c8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user: "find the capital of india then find its current weather". So they want the capital: New Delhi. Then current weather: need up-to-date. We need to use search tool to get current weather. Let\'s search for "current weather in New Delhi".', 'tool_calls': [{'id': 'fc_e8599b61-2b82-4556-9b51-d46671ccf704', 'function': {'arguments': '{"query":"current weather in New Delhi"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 175, 'total_tokens': 261, 'completion_time': 0.11008653, 'completion_tokens_details': {'reasoning_tokens': 55}, 'prompt_time': 0.008637243, 'prompt_tokens_details': None, 'queue_time': 0.345296524, 'total_time': 0.118723773}, 'mode

## How to create custom tool for an agent 
* to create custom tool we need to use a @tool decorator

In [59]:
@tool
def get_weather_data_tool(city:str)->str :
    """Fetch current weather for a city"""

    url = (

        f'http://api.weatherstack.com/current?access_key={WEATHER_API_KEY}&query={city}'
    )

    response = requests.get(url)
    data = response.json()

    return data


In [51]:
tools = [search_tool,get_weather_data_tool]

In [52]:
weather_agent = create_agent(model=llm,tools=tools,system_prompt='you are the weather expert')


In [60]:
response = weather_agent.invoke({"messages":[HumanMessage(content='find the grapes city of india and tell me its current weather')]})


In [61]:
print(response['messages'][-1].content)
response

**City of Grapes in India**

The city famously known as India’s “City of Grapes” is **Nashik** (also spelled Nashik).  
It sits in the foothills of the Western Ghats in Maharashtra and is the country’s primary grape‑growing region, producing roughly 65‑70 % of India’s table grapes and hosting dozens of vineyards and wineries.

---

**Current Weather in Nashik (as of the latest update – Wednesday, 09 Sep 2026)**  

| Parameter | Value | Notes |
|-----------|-------|-------|
| **Temperature** | 22.3 °C (Feels like 21.8 °C) | Mild, pleasant for outdoor activities |
| **Humidity** | 74 % | Moderate humidity |
| **Wind** | 16.2 km/h from the West | Light breeze |
| **UV Index** | 4.6 | Moderate sun exposure |
| **Precipitation** | Patchy rain nearby | 46 % chance of rain; consider carrying an umbrella |
| **Visibility** | 10 km | Clear visibility |

> *Source: Indian Express – Nashik Weather Forecast (September 2026)*

---

**Quick Takeaway**

- **Nashik** is the grape‑capital of India.
- T

{'messages': [HumanMessage(content='find the grapes city of india and tell me its current weather', additional_kwargs={}, response_metadata={}, id='f8f26ab1-3188-4141-8850-13b43ed05ac8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to find the city "Grapes" in India? That seems odd. Maybe "Grapes" is a misspelling or refers to "Grapes" in the context of a city? There\'s a city called "Grapes" maybe "Grapes" is a location in India? Let\'s search.', 'tool_calls': [{'id': 'fc_ea965e28-f746-4a37-8212-d3ea44c144ed', 'function': {'arguments': '{"query":"Grapes city India"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 199, 'total_tokens': 299, 'completion_time': 0.104028354, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.009773908, 'prompt_tokens_details': None, 'queue_time': 0.347395679, 'total_time': 0.113802262}, 'model_name': 'openai/gpt-os